# 01 - Data Preparation

**Purpose:** Combine raw competition data files and create unified train/test datasets.

**Steps:**
1. Mount Google Drive (Colab) or set local paths
2. Load raw competition data files (drv10, drv15, drv30, scoring)
3. Combine training data and create unified target
4. Drop leakage columns
5. Save processed data

**Output:**
- `data/raw/train.csv` - Combined training data with target
- `data/raw/test.csv` - Scoring data (no target)

In [ ]:
# Environment detection
import os
import sys

try:
    import google.colab
    USE_COLAB = True
except ImportError:
    USE_COLAB = False

PROJECT_NAME = '03_Rice_Datathon_Colab'
print(f"Environment: {'Google Colab' if USE_COLAB else 'Local'}")

In [ ]:
# Set up project paths
if USE_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = f'/content/drive/MyDrive/{PROJECT_NAME}'
else:
    PROJECT_ROOT = os.path.dirname(os.getcwd())

sys.path.insert(0, PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path(PROJECT_ROOT) / 'data' / 'raw' / 'competition_data'
OUT_DIR = Path(PROJECT_ROOT) / 'data' / 'raw'

print(f"Raw data dir: {RAW_DIR}")
print(f"Output dir: {OUT_DIR}")

---
## Configuration

Adjust these settings to experiment with different data preparation strategies.

In [ ]:
# =============================================================================
# CONFIGURATION: Leakage columns to drop
# These columns contain target-related information that would cause leakage
# =============================================================================

LEAKAGE_COLS = [
    'REVPAR_2015', 'REVPAR_2020', 'REVPAR_2022', 'REVPAR_2025',
    'REVPAR_GROWTH_2015_2020_PCT', 'REVPAR_GROWTH_2022_2025_PCT',
    'REVPAR_GROWTH_2015_2020_USD', 'REVPAR_GROWTH_2022_2025_USD',
    'QTL_2015_2020_PCT_IN_MARKET', 'QTL_2022_2025_PCT_IN_MARKET'
]

print(f"Leakage columns to drop: {len(LEAKAGE_COLS)}")

---
## Load Raw Data

In [ ]:
print("=" * 60)
print("LOADING RAW DATA")
print("=" * 60)

# Load the 3 drivetime training files
drv10 = pd.read_csv(RAW_DIR / 'master_panel_drv10.csv')
drv15 = pd.read_csv(RAW_DIR / 'master_panel_drv15.csv')
drv30 = pd.read_csv(RAW_DIR / 'master_panel_drv30.csv')

# Load scoring (test) data
scoring = pd.read_csv(RAW_DIR / 'scoring.csv')

print(f"\nData shapes:")
print(f"  drv10: {drv10.shape}")
print(f"  drv15: {drv15.shape}")
print(f"  drv30: {drv30.shape}")
print(f"  scoring: {scoring.shape}")

In [ ]:
# Quick exploration
print("\n--- Data Overview ---")
print(f"\nColumns: {drv10.columns.tolist()[:10]}... ({len(drv10.columns)} total)")
print(f"\ntime_window_tag values: {drv10['time_window_tag'].unique()}")
print(f"trade_area_label values: {drv10['trade_area_label'].unique()}")

---
## Combine Training Data & Create Target

In [ ]:
print("\n--- Combining Training Data ---")

# Combine all training data
train = pd.concat([drv10, drv15, drv30], ignore_index=True)
print(f"Combined train shape: {train.shape}")

# Create unified target column
# Pre-COVID: REVPAR_GROWTH_2015_2020_PCT
# Post-COVID: REVPAR_GROWTH_2022_2025_PCT
train['target'] = np.where(
    train['time_window_tag'] == 'pre',
    train['REVPAR_GROWTH_2015_2020_PCT'],
    train['REVPAR_GROWTH_2022_2025_PCT']
)

print(f"\nTarget statistics:")
print(f"  Mean: {train['target'].mean():.4f}")
print(f"  Std: {train['target'].std():.4f}")
print(f"  Min: {train['target'].min():.4f}")
print(f"  Max: {train['target'].max():.4f}")
print(f"  NaN: {train['target'].isna().sum()}")

In [ ]:
# Create unique row IDs for submission
train['id'] = train['UBID'] + '_' + train['trade_area_label'] + '_' + train['time_window_tag']
scoring['id'] = scoring['UBID'] + '_' + scoring['trade_area_label'] + '_' + scoring['time_window_tag']

print(f"Sample IDs: {train['id'].head(3).tolist()}")

---
## Drop Leakage Columns & Save

In [ ]:
print("\n--- Dropping Leakage Columns ---")

# Drop leakage columns
train_clean = train.drop(columns=LEAKAGE_COLS, errors='ignore')
test_clean = scoring.drop(columns=LEAKAGE_COLS, errors='ignore')

print(f"Train shape after dropping leakage: {train_clean.shape}")
print(f"Test shape after dropping leakage: {test_clean.shape}")

In [ ]:
print("\n--- Saving Processed Data ---")

# Ensure output directory exists
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Save train and test
train_clean.to_csv(OUT_DIR / 'train.csv', index=False)
test_clean.to_csv(OUT_DIR / 'test.csv', index=False)

# Create sample submission
sample_sub = test_clean[['id']].copy()
sample_sub['target'] = 0.0
sample_sub.to_csv(OUT_DIR / 'sample_submission.csv', index=False)

print(f"\nSaved files:")
print(f"  train.csv: {train_clean.shape}")
print(f"  test.csv: {test_clean.shape}")
print(f"  sample_submission.csv: {sample_sub.shape}")

In [ ]:
print("\n" + "=" * 60)
print("DATA PREPARATION COMPLETE")
print("=" * 60)
print(f"\nTrain samples: {len(train_clean):,}")
print(f"Test samples: {len(test_clean):,}")
print(f"Features: {train_clean.shape[1] - 2} (excluding id, target)")
print(f"\n>>> Next: Run 02_preprocessing.ipynb <<<")